Chương 11 – Huấn luyện Mạng thần kinh sâu (Deep Neural Networks)

_Notebook này chứa tất cả mã mẫu và lời giải cho các bài tập trong chương 11._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Thiết lập

Dự án này yêu cầu Python 3.7 trở lên:

In [1]:
import sys

assert sys.version_info >= (3, 7)

Và TensorFlow ≥ 2.8:

In [ ]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

Tương tự như các chương trước, hãy định nghĩa kích thước phông chữ mặc định để làm cho các hình vẽ đẹp hơn:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Và hãy tạo thư mục `images/deep` (nếu nó chưa tồn tại), và định nghĩa hàm `save_fig()` được sử dụng xuyên suốt notebook này để lưu các hình vẽ với độ phân giải cao cho sách:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "deep"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

# Vấn đề Triệt tiêu/Bùng nổ Độ dốc

In [ ]:
# extra code – this cell generates and saves Figure 11–1

import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-5, 5, 200)

plt.plot([-5, 5], [0, 0], 'k-')
plt.plot([-5, 5], [1, 1], 'k--')
plt.plot([0, 0], [-0.2, 1.2], 'k-')
plt.plot([-5, 5], [-3/4, 7/4], 'g--')
plt.plot(z, sigmoid(z), "b-", linewidth=2,
         label=r"$\sigma(z) = \dfrac{1}{1+e^{-z}}$")
props = dict(facecolor='black', shrink=0.1)
plt.annotate('Saturating', xytext=(3.5, 0.7), xy=(5, 1), arrowprops=props,
             fontsize=14, ha="center")
plt.annotate('Saturating', xytext=(-3.5, 0.3), xy=(-5, 0), arrowprops=props,
             fontsize=14, ha="center")
plt.annotate('Linear', xytext=(2, 0.2), xy=(0, 0.5), arrowprops=props,
             fontsize=14, ha="center")
plt.grid(True)
plt.axis([-5, 5, -0.2, 1.2])
plt.xlabel("$z$")
plt.legend(loc="upper left", fontsize=16)

save_fig("sigmoid_saturation_plot")
plt.show()

## Khởi tạo Xavier và He

In [ ]:
dense = tf.keras.layers.Dense(50, activation="relu",
                              kernel_initializer="he_normal")

In [ ]:
he_avg_init = tf.keras.initializers.VarianceScaling(scale=2., mode="fan_avg",
                                                    distribution="uniform")
dense = tf.keras.layers.Dense(50, activation="sigmoid",
                              kernel_initializer=he_avg_init)

## Hàm Kích hoạt Không Bão hòa

### Leaky ReLU

In [ ]:
# extra code – this cell generates and saves Figure 11–2

def leaky_relu(z, alpha):
    return np.maximum(alpha * z, z)

z = np.linspace(-5, 5, 200)
plt.plot(z, leaky_relu(z, 0.1), "b-", linewidth=2, label=r"$LeakyReLU(z) = max(\alpha z, z)$")
plt.plot([-5, 5], [0, 0], 'k-')
plt.plot([0, 0], [-1, 3.7], 'k-')
plt.grid(True)
props = dict(facecolor='black', shrink=0.1)
plt.annotate('Leak', xytext=(-3.5, 0.5), xy=(-5, -0.3), arrowprops=props,
             fontsize=14, ha="center")
plt.xlabel("$z$")
plt.axis([-5, 5, -1, 3.7])
plt.gca().set_aspect("equal")
plt.legend()

save_fig("leaky_relu_plot")
plt.show()

In [ ]:
leaky_relu = tf.keras.layers.LeakyReLU(alpha=0.2)  # defaults to alpha=0.3
dense = tf.keras.layers.Dense(50, activation=leaky_relu,
                              kernel_initializer="he_normal")

In [ ]:
model = tf.keras.models.Sequential([
    # [...]  # more layers
    tf.keras.layers.Dense(50, kernel_initializer="he_normal"),  # no activation
    tf.keras.layers.LeakyReLU(alpha=0.2),  # activation as a separate layer
    # [...]  # more layers
])

### ELU

Việc triển khai ELU trong TensorFlow là rất đơn giản, chỉ cần chỉ định hàm kích hoạt khi xây dựng mỗi lớp, và sử dụng khởi tạo He:

In [ ]:
dense = tf.keras.layers.Dense(50, activation="elu",
                              kernel_initializer="he_normal")

### SELU

Theo mặc định, các siêu tham số của SELU (`scale` và `alpha`) được điều chỉnh theo cách sao cho giá trị trung bình đầu ra của mỗi nơ-ron vẫn gần bằng 0, và độ lệch chuẩn vẫn gần bằng 1 (giả sử các đầu vào cũng được chuẩn hóa với giá trị trung bình 0 và độ lệch chuẩn 1, và các ràng buộc khác được tôn trọng, như đã giải thích trong sách). Sử dụng hàm kích hoạt này, ngay cả mạng nơ-ron sâu 1.000 lớp cũng giữ được xấp xỉ giá trị trung bình 0 và độ lệch chuẩn 1 qua tất cả các lớp, tránh được vấn đề bùng nổ/triệt tiêu độ dốc:

In [ ]:
# extra code – this cell generates and saves Figure 11–3

from scipy.special import erfc

# alpha and scale to self normalize with mean 0 and standard deviation 1
# (see equation 14 in the paper):
alpha_0_1 = -np.sqrt(2 / np.pi) / (erfc(1 / np.sqrt(2)) * np.exp(1 / 2) - 1)
scale_0_1 = (
    (1 - erfc(1 / np.sqrt(2)) * np.sqrt(np.e))
    * np.sqrt(2 * np.pi)
    * (
        2 * erfc(np.sqrt(2)) * np.e ** 2
        + np.pi * erfc(1 / np.sqrt(2)) ** 2 * np.e
        - 2 * (2 + np.pi) * erfc(1 / np.sqrt(2)) * np.sqrt(np.e)
        + np.pi
        + 2
    ) ** (-1 / 2)
)

def elu(z, alpha=1):
    return np.where(z < 0, alpha * (np.exp(z) - 1), z)

def selu(z, scale=scale_0_1, alpha=alpha_0_1):
    return scale * elu(z, alpha)

z = np.linspace(-5, 5, 200)
plt.plot(z, elu(z), "b-", linewidth=2, label=r"ELU$_\alpha(z) = \alpha (e^z - 1)$ if $z < 0$, else $z$")
plt.plot(z, selu(z), "r--", linewidth=2, label=r"SELU$(z) = 1.05 \, $ELU$_{1.67}(z)$")
plt.plot([-5, 5], [0, 0], 'k-')
plt.plot([-5, 5], [-1, -1], 'k:', linewidth=2)
plt.plot([-5, 5], [-1.758, -1.758], 'k:', linewidth=2)
plt.plot([0, 0], [-2.2, 3.2], 'k-')
plt.grid(True)
plt.axis([-5, 5, -2.2, 3.2])
plt.xlabel("$z$")
plt.gca().set_aspect("equal")
plt.legend()

save_fig("elu_selu_plot")
plt.show()

Sử dụng SELU rất đơn giản:

In [ ]:
dense = tf.keras.layers.Dense(50, activation="selu",
                              kernel_initializer="lecun_normal")

**Tài liệu bổ sung – một ví dụ về mạng được tự điều chuẩn (self-regularized network) sử dụng SELU**

Hãy tạo một mạng nơ-ron cho Fashion MNIST với 100 lớp ẩn, sử dụng hàm kích hoạt SELU:

In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[28, 28]))
for layer in range(100):
    model.add(tf.keras.layers.Dense(100, activation="selu",
                                    kernel_initializer="lecun_normal"))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              metrics=["accuracy"])

Bây giờ hãy huấn luyện nó. Đừng quên **chuẩn hóa (scale)** các đầu vào về giá trị trung bình 0 và độ lệch chuẩn 1:

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]
X_train, X_valid, X_test = X_train / 255, X_valid / 255, X_test / 255

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
pixel_means = X_train.mean(axis=0, keepdims=True)
pixel_stds = X_train.std(axis=0, keepdims=True)
X_train_scaled = (X_train - pixel_means) / pixel_stds
X_valid_scaled = (X_valid - pixel_means) / pixel_stds
X_test_scaled = (X_test - pixel_means) / pixel_stds

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=5,
                    validation_data=(X_valid_scaled, y_valid))

Mạng đã **học** được, mặc dù nó **sâu** đến như vậy. Bây giờ hãy xem điều gì xảy ra nếu chúng ta thử sử dụng hàm kích hoạt **ReLU** thay thế:

In [ ]:
tf.random.set_seed(42)

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[28, 28]))
for layer in range(100):
    model.add(tf.keras.layers.Dense(100, activation="relu",
                                    kernel_initializer="he_normal"))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
              metrics=["accuracy"])

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=5,
                    validation_data=(X_valid_scaled, y_valid))

Hoàn toàn không tốt chút nào, chúng ta đã gặp phải vấn đề **triệt tiêu/bùng nổ độ dốc**.

### GELU, Swish và Mish

In [ ]:
# extra code – this cell generates and saves Figure 11–4

def swish(z, beta=1):
    return z * sigmoid(beta * z)

def approx_gelu(z):
    return swish(z, beta=1.702)

def softplus(z):
    return np.log(1 + np.exp(z))

def mish(z):
    return z * np.tanh(softplus(z))

z = np.linspace(-4, 2, 200)

beta = 0.6
plt.plot(z, approx_gelu(z), "b-", linewidth=2,
         label=r"GELU$(z) = z\,\Phi(z)$")
plt.plot(z, swish(z), "r--", linewidth=2,
         label=r"Swish$(z) = z\,\sigma(z)$")
plt.plot(z, swish(z, beta), "r:", linewidth=2,
         label=fr"Swish$_{{\beta={beta}}}(z)=z\,\sigma({beta}\,z)$")
plt.plot(z, mish(z), "g:", linewidth=3,
         label=fr"Mish$(z) = z\,\tanh($softplus$(z))$")
plt.plot([-4, 2], [0, 0], 'k-')
plt.plot([0, 0], [-2.2, 3.2], 'k-')
plt.grid(True)
plt.axis([-4, 2, -1, 2])
plt.gca().set_aspect("equal")
plt.xlabel("$z$")
plt.legend(loc="upper left")

save_fig("gelu_swish_mish_plot")
plt.show()

# Chuẩn hóa Hàng loạt (Batch Normalization)

In [ ]:
# extra code - clear the name counters and set the random seed
tf.keras.backend.clear_session()
tf.random.set_seed(42)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(300, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()

In [ ]:
[(var.name, var.trainable) for var in model.layers[1].variables]

In [ ]:
# extra code – just show that the model works! 😊
# model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
#               metrics="accuracy")
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Đôi khi áp dụng **BN (Batch Normalization)** trước hàm kích hoạt hoạt động tốt hơn (có một cuộc tranh luận về chủ đề này). Hơn nữa, lớp đứng trước một lớp `BatchNormalization` không cần phải có các **hệ số thiên vị (bias terms)**, vì lớp `BatchNormalization` cũng có một số, điều đó sẽ là sự lãng phí tham số, do đó bạn có thể đặt `use_bias=False` khi tạo các lớp đó:

In [ ]:
# extra code - clear the name counters and set the random seed
tf.keras.backend.clear_session()
tf.random.set_seed(42)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, kernel_initializer="he_normal", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(100, kernel_initializer="he_normal", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
# extra code – just show that the model works! 😊
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

## Cắt bớt Độ dốc (Gradient Clipping)

Tất cả các `tf.keras.optimizers` đều chấp nhận các đối số `clipnorm` hoặc `clipvalue`:

In [ ]:
optimizer = tf.keras.optimizers.SGD(clipvalue=1.0)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer)

In [ ]:
optimizer = tf.keras.optimizers.SGD(clipnorm=1.0)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer)

## Tái sử dụng các Lớp đã được Huấn luyện trước (Reusing Pretrained Layers)

### Tái sử dụng một mô hình Keras

Hãy chia tập huấn luyện fashion MNIST thành hai phần:
* `X_train_A`: tất cả các hình ảnh của tất cả các mặt hàng ngoại trừ áo phông/áo top và áo len chui đầu (các lớp 0 và 2).
* `X_train_B`: một tập huấn luyện nhỏ hơn nhiều chỉ gồm 200 hình ảnh đầu tiên của áo phông/áo top và áo len chui đầu.

Tập xác thực (validation set) và tập kiểm tra (test set) cũng được chia theo cách này, nhưng không giới hạn số lượng hình ảnh.

Chúng ta sẽ huấn luyện một mô hình trên tập A (nhiệm vụ phân loại với 8 lớp), và cố gắng tái sử dụng nó để giải quyết tập B (phân loại nhị phân). Chúng ta hy vọng truyền một chút kiến thức từ nhiệm vụ A sang nhiệm vụ B, vì các lớp trong tập A (quần dài, váy, áo khoác, sandal, áo sơ mi, giày thể thao, túi xách và bốt cổ chân) có phần nào tương tự với các lớp trong tập B (áo phông/áo top và áo len chui đầu). Tuy nhiên, vì chúng ta đang sử dụng các lớp `Dense`, chỉ những mẫu xuất hiện ở cùng một vị trí mới có thể được tái sử dụng (ngược lại, các lớp tích chập sẽ truyền tốt hơn nhiều, vì các mẫu đã học có thể được phát hiện ở bất cứ đâu trên hình ảnh, như chúng ta sẽ thấy trong chương 14).

In [ ]:
# extra code – split Fashion MNIST into tasks A and B, then train and save
#              model A to "my_model_A".

pos_class_id = class_names.index("Pullover")
neg_class_id = class_names.index("T-shirt/top")

def split_dataset(X, y):
    y_for_B = (y == pos_class_id) | (y == neg_class_id)
    y_A = y[~y_for_B]
    y_B = (y[y_for_B] == pos_class_id).astype(np.float32)
    old_class_ids = list(set(range(10)) - set([neg_class_id, pos_class_id]))
    for old_class_id, new_class_id in zip(old_class_ids, range(8)):
        y_A[y_A == old_class_id] = new_class_id  # reorder class ids for A
    return ((X[~y_for_B], y_A), (X[y_for_B], y_B))

(X_train_A, y_train_A), (X_train_B, y_train_B) = split_dataset(X_train, y_train)
(X_valid_A, y_valid_A), (X_valid_B, y_valid_B) = split_dataset(X_valid, y_valid)
(X_test_A, y_test_A), (X_test_B, y_test_B) = split_dataset(X_test, y_test)
X_train_B = X_train_B[:200]
y_train_B = y_train_B[:200]

tf.random.set_seed(42)

model_A = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(8, activation="softmax")
])

model_A.compile(loss="sparse_categorical_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=["accuracy"])
history = model_A.fit(X_train_A, y_train_A, epochs=20,
                      validation_data=(X_valid_A, y_valid_A))
#model_A.save("my_model_A")
model_A.save("my_model_A.keras")


In [ ]:
# extra code – train and evaluate model B, without reusing model A

tf.random.set_seed(42)
model_B = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_B.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=["accuracy"])
history = model_B.fit(X_train_B, y_train_B, epochs=20,
                      validation_data=(X_valid_B, y_valid_B))
model_B.evaluate(X_test_B, y_test_B)

Mô hình B đạt độ chính xác 91.85% trên tập kiểm tra. Bây giờ hãy thử tái sử dụng mô hình A đã được huấn luyện trước.

In [ ]:
#model_A = tf.keras.models.load_model("my_model_A")
model_A = tf.keras.models.load_model("my_model_A.keras")

model_B_on_A = tf.keras.Sequential(model_A.layers[:-1])
model_B_on_A.add(tf.keras.layers.Dense(1, activation="sigmoid"))

Lưu ý rằng `model_B_on_A` và `model_A` hiện tại thực sự **chia sẻ các lớp (share layers)**, vì vậy khi chúng ta huấn luyện một mô hình, nó sẽ cập nhật cả hai mô hình. Nếu chúng ta muốn tránh điều đó, chúng ta cần xây dựng `model_B_on_A` dựa trên một **bản sao (clone)** của `model_A`:

In [ ]:
tf.random.set_seed(42)  # extra code – ensure reproducibility

In [ ]:
model_A_clone = tf.keras.models.clone_model(model_A)
model_A_clone.set_weights(model_A.get_weights())

In [ ]:
# extra code – creating model_B_on_A just like in the previous cell
model_B_on_A = tf.keras.Sequential(model_A_clone.layers[:-1])
model_B_on_A.add(tf.keras.layers.Dense(1, activation="sigmoid"))

In [ ]:
for layer in model_B_on_A.layers[:-1]:
    layer.trainable = False

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy", optimizer=optimizer,
                     metrics=["accuracy"])

In [ ]:
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=4,
                           validation_data=(X_valid_B, y_valid_B))

for layer in model_B_on_A.layers[:-1]:
    layer.trainable = True

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B_on_A.compile(loss="binary_crossentropy", optimizer=optimizer,
                     metrics=["accuracy"])
history = model_B_on_A.fit(X_train_B, y_train_B, epochs=16,
                           validation_data=(X_valid_B, y_valid_B))

Vậy, **phán quyết cuối cùng** là gì?

In [ ]:
model_B_on_A.evaluate(X_test_B, y_test_B)

Tuyệt vời! Chúng ta đã có được một chút **chuyển giao (transfer)**: độ chính xác của mô hình đã tăng thêm 2 điểm phần trăm, từ 91.85% lên 93.85%. Điều này có nghĩa là **tỷ lệ lỗi (error rate)** đã giảm gần 25%:

In [ ]:
1 - (100 - 93.85) / (100 - 91.85)

# Các Trình Tối ưu hóa Nhanh hơn (Faster Optimizers)

In [ ]:
# extra code – a little function to test an optimizer on Fashion MNIST

def build_model(seed=42):
    tf.random.set_seed(seed)
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[28, 28]),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

def build_and_train_model(optimizer):
    model = build_model()
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])
    return model.fit(X_train, y_train, epochs=10,
                     validation_data=(X_valid, y_valid))

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

In [ ]:
history_sgd = build_and_train_model(optimizer)  # extra code

## Tối ưu hóa Momentum

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

In [ ]:
history_momentum = build_and_train_model(optimizer)  # extra code

## Độ dốc Tăng tốc Nesterov (Nesterov Accelerated Gradient)

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9,
                                    nesterov=True)

In [ ]:
history_nesterov = build_and_train_model(optimizer)  # extra code

## AdaGrad

In [ ]:
optimizer = tf.keras.optimizers.Adagrad(learning_rate=0.001)

In [ ]:
history_adagrad = build_and_train_model(optimizer)  # extra code

## RMSProp

In [ ]:
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001, rho=0.9)

In [ ]:
history_rmsprop = build_and_train_model(optimizer)  # extra code

## Tối ưu hóa Adam

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9,
                                     beta_2=0.999)

In [ ]:
history_adam = build_and_train_model(optimizer)  # extra code

**Tối ưu hóa Adamax**

In [ ]:
optimizer = tf.keras.optimizers.Adamax(learning_rate=0.001, beta_1=0.9,
                                       beta_2=0.999)

In [ ]:
history_adamax = build_and_train_model(optimizer)  # extra code

**Tối ưu hóa Nadam**

In [ ]:
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,
                                      beta_2=0.999)

In [ ]:
history_nadam = build_and_train_model(optimizer)  # extra code

**Tối ưu hóa AdamW**

Lưu ý: Kể từ TF 1.12, `AdamW` không còn là thử nghiệm nữa. Nó có sẵn tại `tf.keras.optimizers.AdamW` thay vì `tf.keras.optimizers.experimental.AdamW`.

In [ ]:
optimizer = tf.keras.optimizers.AdamW(weight_decay=1e-5, learning_rate=0.001,
                                      beta_1=0.9, beta_2=0.999)

In [ ]:
history_adamw = build_and_train_model(optimizer)  # extra code

In [ ]:
# extra code – visualize the learning curves of all the optimizers

for loss in ("loss", "val_loss"):
    plt.figure(figsize=(12, 8))
    opt_names = "SGD Momentum Nesterov AdaGrad RMSProp Adam Adamax Nadam AdamW"
    for history, opt_name in zip((history_sgd, history_momentum, history_nesterov,
                                  history_adagrad, history_rmsprop, history_adam,
                                  history_adamax, history_nadam, history_adamw),
                                 opt_names.split()):
        plt.plot(history.history[loss], label=f"{opt_name}", linewidth=3)

    plt.grid()
    plt.xlabel("Epochs")
    plt.ylabel({"loss": "Training loss", "val_loss": "Validation loss"}[loss])
    plt.legend(loc="upper left")
    plt.axis([0, 9, 0.1, 0.7])
    plt.show()

## Lập lịch Tốc độ Học (Learning Rate Scheduling)

### Lập lịch Theo Hàm Mũ (Power Scheduling)

```python

learning_rate = initial_learning_rate / (1 + step / decay_steps)**power

```



Keras dùng `power = 1`.

**Lưu ý**: Đối số `decay` trong các **trình tối ưu hóa (optimizers)** đã bị **khai tử (deprecated)**. Các trình tối ưu hóa cũ có triển khai đối số `decay` vẫn có sẵn trong `tf.keras.optimizers.legacy`, nhưng thay vào đó, bạn nên sử dụng các **bộ lập lịch (schedulers)** trong `tf.keras.optimizers.schedules`.

In [ ]:
# DEPRECATED:
#optimizer = tf.keras.optimizers.legacy.SGD(learning_rate=0.01, decay=1e-4)
# RECOMMENDED:
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.01,
    decay_steps=10_000,
    decay_rate=1.0,
    staircase=False
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)


In [ ]:
# RECOMMENDED:
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=0.01,
    decay_steps=10_000,
    decay_rate=1.0,
    staircase=False
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

Bộ lập lịch `InverseTimeDecay` sử dụng công thức `learning_rate = initial_learning_rate / (1 + decay_rate * step / decay_steps)`. Nếu bạn đặt `staircase=True`, thì nó sẽ thay thế `step / decay_step` bằng `floor(step / decay_step)`.

In [ ]:
history_power_scheduling = build_and_train_model(optimizer)  # extra code

In [ ]:
# extra code – this cell plots power scheduling with staircase=True or False

initial_learning_rate = 0.01
decay_rate = 1.0
decay_steps = 10_000

steps = np.arange(100_000)
lrs = initial_learning_rate / (1 + decay_rate * steps / decay_steps)
lrs2 = initial_learning_rate / (1 + decay_rate * np.floor(steps / decay_steps))

plt.plot(steps, lrs,  "-", label="staircase=False")
plt.plot(steps, lrs2,  "-", label="staircase=True")
plt.axis([0, steps.max(), 0, 0.0105])
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Power Scheduling", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

### Lập lịch Theo Hàm Số Mũ (Exponential Scheduling)

```python
learning_rate = initial_learning_rate * decay_rate ** (step / decay_steps)
```

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=20_000,
    decay_rate=0.1,
    staircase=False
)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)

In [ ]:
history_exponential_scheduling = build_and_train_model(optimizer)  # extra code

In [ ]:
# extra code – this cell plots exponential scheduling

initial_learning_rate = 0.01
decay_rate = 0.1
decay_steps = 20_000

steps = np.arange(100_000)
lrs = initial_learning_rate * decay_rate ** (steps / decay_steps)
lrs2 = initial_learning_rate * decay_rate ** np.floor(steps / decay_steps)

plt.plot(steps, lrs,  "-", label="staircase=False")
plt.plot(steps, lrs2,  "-", label="staircase=True")
plt.axis([0, steps.max(), 0, 0.0105])
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Exponential Scheduling", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

Keras cũng cung cấp lớp callback **`LearningRateScheduler`** cho phép bạn định nghĩa hàm lập lịch của riêng mình. Hãy xem cách bạn có thể sử dụng nó để triển khai **phân rã hàm số mũ (exponential decay)**. Lưu ý rằng trong trường hợp này, tốc độ học **chỉ thay đổi sau mỗi epoch**, chứ không phải sau mỗi bước:

In [ ]:
def exponential_decay_fn(epoch):
    return 0.01 * 0.1 ** (epoch / 20)

In [ ]:
def exponential_decay(lr0, s):
    def exponential_decay_fn(epoch):
        return lr0 * 0.1 ** (epoch / s)
    return exponential_decay_fn

exponential_decay_fn = exponential_decay(lr0=0.01, s=20)

In [ ]:
# extra code – build and compile a model for Fashion MNIST

tf.random.set_seed(42)
model = build_model()
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])

In [ ]:
n_epochs = 20

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(exponential_decay_fn)
history = model.fit(X_train, y_train, epochs=n_epochs,
                    validation_data=(X_valid, y_valid),
                    callbacks=[lr_scheduler])

Hoặc, hàm lập lịch có thể nhận **tốc độ học hiện tại** làm đối số thứ hai:

In [ ]:
def exponential_decay_fn(epoch, lr):
    return lr * 0.1 ** (1 / 20)

**Tài liệu bổ sung**: nếu bạn muốn sử dụng một hàm lập lịch tùy chỉnh cập nhật **tốc độ học** sau **mỗi lần lặp (iteration)** thay vì sau **mỗi epoch**, bạn có thể viết lớp **callback** của riêng mình như sau:

In [ ]:
# K = tf.keras.backend

# class ExponentialDecay(tf.keras.callbacks.Callback):
#     def __init__(self, n_steps=40_000):
#         super().__init__()
#         self.n_steps = n_steps

#     def on_batch_begin(self, batch, logs=None):
#         # Note: the `batch` argument is reset at each epoch
#         lr = K.get_value(self.model.optimizer.learning_rate)
#         new_learning_rate = lr * 0.1 ** (1 / self.n_steps)
#         K.set_value(self.model.optimizer.learning_rate, new_learning_rate)

#     def on_epoch_end(self, epoch, logs=None):
#         logs = logs or {}
#         logs['lr'] = K.get_value(self.model.optimizer.learning_rate)
import tensorflow as tf
from tensorflow.keras import backend as K

class CustomLRScheduler(tf.keras.callbacks.Callback):
    def __init__(self, n_steps=40000, decay_factor=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.decay_factor = decay_factor

    def on_batch_begin(self, batch, logs=None):
        lr = float(self.model.optimizer.learning_rate.numpy())
        new_lr = lr * (self.decay_factor ** (1 / self.n_steps))
        self.model.optimizer.learning_rate.assign(new_lr)  # ✅ dùng assign thay vì K.set_value

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['learning_rate'] = float(self.model.optimizer.learning_rate.numpy())



In [ ]:
# ✅ Tạo model mới
import tensorflow as tf
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Flatten(input_shape=[28, 28]),
    layers.Dense(100, activation="relu"),
    layers.Dense(100, activation="relu"),
    layers.Dense(10, activation="softmax")
])

# ✅ compile với optimizer object
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

n_epochs = 20
batch_size = 32
n_steps = n_epochs * (len(X_train) // batch_size)
lr_scheduler = CustomLRScheduler(n_steps=n_steps)

history = model.fit(
    X_train, y_train,
    epochs=n_epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[lr_scheduler]
)


callback tương thích với Keras 3

In [ ]:
import math
import tensorflow as tf

class ExponentialDecay(tf.keras.callbacks.Callback):
    def __init__(self, n_steps=40_000):
        super().__init__()
        self.n_steps = n_steps

    def on_batch_begin(self, batch, logs=None):
        # ✅ Lấy learning_rate hiện tại
        lr = float(self.model.optimizer.learning_rate.numpy())
        # ✅ Tính LR mới
        new_learning_rate = lr * 0.1 ** (1 / self.n_steps)
        # ✅ Gán lại giá trị LR
        self.model.optimizer.learning_rate.assign(new_learning_rate)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['lr'] = float(self.model.optimizer.learning_rate.numpy())

Compile model đúng cách

In [ ]:
# KHÔNG được dùng optimizer="sgd"
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)


Huấn luyện

In [ ]:
batch_size = 32
n_epochs = 20
n_steps = n_epochs * math.ceil(len(X_train) / batch_size)
exp_decay = ExponentialDecay(n_steps)

history = model.fit(
    X_train, y_train,
    epochs=n_epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[exp_decay]
)


### Lập lịch Hằng số Từng phần (Piecewise Constant Scheduling)

In [ ]:
# lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
#     boundaries=[50_000, 80_000],
#     values=[0.01, 0.005, 0.001]
# )
# optimizer = tf.keras.optimizers.SGD(learning_rate=lr_schedule)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_and_train_model(optimizer, n_epochs=20, batch_size=32):
    # 1. Xây dựng model
    model = models.Sequential([
        layers.Flatten(input_shape=[28, 28]),
        layers.Dense(100, activation="relu"),
        layers.Dense(100, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    # 2. Compile với optimizer truyền vào
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )

    # 3. Huấn luyện model
    history = model.fit(
        X_train, y_train,
        epochs=n_epochs,
        validation_data=(X_valid, y_valid),
        batch_size=batch_size
    )

    return history


In [ ]:
#sử dụng
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
history_piecewise_scheduling = build_and_train_model(optimizer)


In [ ]:
# extra code – this cell plots piecewise constant scheduling
import numpy as np
import matplotlib.pyplot as plt

boundaries = [50_000, 80_000]
values = [0.01, 0.005, 0.001]

steps = np.arange(100_000)

lrs = np.full(len(steps), values[0])
for boundary, value in zip(boundaries, values[1:]):
    lrs[boundary:] = value

plt.plot(steps, lrs, "-")
plt.axis([0, steps.max(), 0, 0.0105])
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Piecewise Constant Scheduling", fontsize=14)
plt.grid(True)
plt.show()

Giống như cách chúng ta đã làm với lập lịch theo hàm số mũ, chúng ta cũng có thể triển khai **lập lịch hằng số từng phần** theo cách thủ công:

In [ ]:
def piecewise_constant_fn(epoch):
    if epoch < 5:
        return 0.01
    elif epoch < 15:
        return 0.005
    else:
        return 0.001

In [ ]:
# extra code – this cell demonstrates a more general way to define
#              piecewise constant scheduling.

def piecewise_constant(boundaries, values):
    boundaries = np.array([0] + boundaries)
    values = np.array(values)
    def piecewise_constant_fn(epoch):
        return values[(boundaries > epoch).argmax() - 1]
    return piecewise_constant_fn

piecewise_constant_fn = piecewise_constant([5, 15], [0.01, 0.005, 0.001])

định nghĩa đơn giản cho build_model()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential([
        layers.Flatten(input_shape=[28, 28]),
        layers.Dense(100, activation="relu"),
        layers.Dense(100, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    return model


Sử dụng

In [ ]:
# Hàm điều chỉnh LR theo epoch
def piecewise_constant_fn(epoch):
    if epoch < 5:
        return 0.01
    elif epoch < 15:
        return 0.005
    else:
        return 0.001

n_epochs = 25
lr0 = 0.01  # learning rate khởi đầu

# Callback scheduler
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(piecewise_constant_fn)

# Xây dựng và train model
model = build_model()
optimizer = tf.keras.optimizers.Nadam(learning_rate=lr0)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=n_epochs,
    validation_data=(X_valid, y_valid),
    callbacks=[lr_scheduler]
)


Chúng ta đã xem xét `InverseTimeDecay`, `ExponentialDecay` và `PiecewiseConstantDecay`. Một vài bộ lập lịch khác có sẵn trong `tf.keras.optimizers.schedules`, đây là danh sách đầy đủ:

In [ ]:
for name in sorted(dir(tf.keras.optimizers.schedules)):
    if name[0] == name[0].lower():  # must start with capital letter
        continue
    scheduler_class = getattr(tf.keras.optimizers.schedules, name)
    print(f"• {name} – {scheduler_class.__doc__.splitlines()[0]}")

### Lập lịch Theo Hiệu suất (Performance Scheduling)

In [ ]:
# extra code – build and compile the model

model = build_model()
optimizer = tf.keras.optimizers.SGD(learning_rate=lr0)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
history = model.fit(X_train, y_train, epochs=n_epochs,
                    validation_data=(X_valid, y_valid),
                    callbacks=[lr_scheduler])

In [ ]:
# extra code – this cell plots performance scheduling

#plt.plot(history.epoch, history.history["lr"], "bo-")
plt.plot(history.epoch, history.history["learning_rate"], "bo-")

plt.xlabel("Epoch")
plt.ylabel("Learning Rate", color='b')
plt.tick_params('y', colors='b')
plt.gca().set_xlim(0, n_epochs - 1)
plt.grid(True)

ax2 = plt.gca().twinx()
ax2.plot(history.epoch, history.history["val_loss"], "r^-")
ax2.set_ylabel('Validation Loss', color='r')
ax2.tick_params('y', colors='r')

plt.title("Reduce LR on Plateau", fontsize=14)
plt.show()

### Lập lịch 1 Chu kỳ (1Cycle scheduling)

Callback tùy chỉnh **`ExponentialLearningRate`** cập nhật **tốc độ học** trong quá trình huấn luyện, ở cuối **mỗi batch**. Nó nhân tốc độ học với một hằng số `factor`. Nó cũng lưu lại tốc độ học và độ lỗi ở mỗi batch. Vì `logs["loss"]` thực chất là **độ lỗi trung bình** kể từ đầu **epoch**, và chúng ta muốn lưu độ lỗi của **batch** hiện tại, nên chúng ta phải tính **giá trị trung bình nhân với số lượng batch** kể từ đầu epoch để có được tổng độ lỗi cho đến nay, sau đó chúng ta trừ đi tổng độ lỗi ở batch trước để có được độ lỗi của batch hiện tại.

In [ ]:
K = tf.keras.backend

class ExponentialLearningRate(tf.keras.callbacks.Callback):
    def __init__(self, factor):
        self.factor = factor
        self.rates = []
        self.losses = []

    def on_epoch_begin(self, epoch, logs=None):
        self.sum_of_epoch_losses = 0

    def on_batch_end(self, batch, logs=None):
        mean_epoch_loss = logs["loss"]  # the epoch's mean loss so far
        new_sum_of_epoch_losses = mean_epoch_loss * (batch + 1)
        batch_loss = new_sum_of_epoch_losses - self.sum_of_epoch_losses
        self.sum_of_epoch_losses = new_sum_of_epoch_losses
        self.rates.append(K.get_value(self.model.optimizer.learning_rate))
        self.losses.append(batch_loss)
        K.set_value(self.model.optimizer.learning_rate,
                    self.model.optimizer.learning_rate * self.factor)

Hàm `find_learning_rate()` huấn luyện mô hình bằng cách sử dụng callback **`ExponentialLearningRate`**, và nó trả về **tốc độ học** và **độ lỗi batch** tương ứng. Cuối cùng, nó **khôi phục (restores)** mô hình và **bộ tối ưu hóa (optimizer)** của nó về trạng thái ban đầu.

In [ ]:
def find_learning_rate(model, X, y, epochs=1, batch_size=32, min_rate=1e-4,
                       max_rate=1):
    init_weights = model.get_weights()
    iterations = math.ceil(len(X) / batch_size) * epochs
    factor = (max_rate / min_rate) ** (1 / iterations)
    init_lr = K.get_value(model.optimizer.learning_rate)
    K.set_value(model.optimizer.learning_rate, min_rate)
    exp_lr = ExponentialLearningRate(factor)
    history = model.fit(X, y, epochs=epochs, batch_size=batch_size,
                        callbacks=[exp_lr])
    K.set_value(model.optimizer.learning_rate, init_lr)
    model.set_weights(init_weights)
    return exp_lr.rates, exp_lr.losses

Hàm `plot_lr_vs_loss()` vẽ biểu đồ **tốc độ học** so với **độ lỗi**. **Tốc độ học tối ưu** để sử dụng làm tốc độ học tối đa trong **1 chu kỳ (1cycle)** nằm gần điểm thấp nhất của đường cong.

In [ ]:
def plot_lr_vs_loss(rates, losses):
    plt.plot(rates, losses, "b")
    plt.gca().set_xscale('log')
    max_loss = losses[0] + min(losses)
    plt.hlines(min(losses), min(rates), max(rates), color="k")
    plt.axis([min(rates), max(rates), 0, max_loss])
    plt.xlabel("Learning rate")
    plt.ylabel("Loss")
    plt.grid()

Hãy xây dựng một mô hình Fashion MNIST đơn giản và biên dịch nó:

In [ ]:
# model = build_model()
# # model.compile(loss="sparse_categorical_crossentropy",
# #               optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
# #               metrics=["accuracy"])
import tensorflow as tf
from tensorflow.keras import layers, models

# Tạo mô hình đơn giản (ví dụ cho Fashion MNIST)
def build_model():
    model = models.Sequential([
        layers.Flatten(input_shape=[28, 28]),
        layers.Dense(100, activation="relu"),
        layers.Dense(100, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    return model

# Khởi tạo optimizer đúng chuẩn
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)

# Biên dịch mô hình
model = build_model()
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)



In [ ]:
# ✅ Bước 2: nếu find_learning_rate vẫn dùng K.set_value thì ép kiểu
import numpy as np

def lr_finder(model, X, y, epochs=1, batch_size=128, min_lr=1e-5, max_lr=10):
    num_batches = len(X) // batch_size * epochs
    lr_mult = (max_lr / min_lr) ** (1 / num_batches)
    lrs = []
    losses = []

    # Đặt LR khởi đầu
    model.optimizer.learning_rate.assign(min_lr)

    for epoch in range(epochs):
        for i in range(0, len(X), batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]

            logs = model.train_on_batch(X_batch, y_batch)
            loss = logs if np.isscalar(logs) else logs[0]
            losses.append(loss)
            lrs.append(float(model.optimizer.learning_rate.numpy()))

            # tăng LR theo cấp số nhân
            model.optimizer.learning_rate.assign(model.optimizer.learning_rate * lr_mult)

    return np.array(lrs), np.array(losses)


Bây giờ hãy tìm **tốc độ học tối đa tối ưu** cho **1 chu kỳ (1cycle)**:

In [ ]:
# batch_size = 128
# rates, losses = find_learning_rate(model, X_train, y_train, epochs=1,
#                                    batch_size=batch_size)
# plot_lr_vs_loss(rates, losses)
# ✅ Bước 3: chạy LR finder
import matplotlib.pyplot as plt

# Chạy LR Finder
batch_size = 128
rates, losses = lr_finder(model, X_train, y_train, epochs=1, batch_size=batch_size)

# Vẽ Learning Rate vs Loss
plt.figure(figsize=(10,6))
plt.plot(rates, losses)
plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.title("Learning Rate Finder")
plt.grid(True)
plt.show()


Có vẻ như tốc độ học tối đa để sử dụng cho **1 chu kỳ (1cycle)** là khoảng $10^{-1}$.

Callback tùy chỉnh **`OneCycleScheduler`** cập nhật **tốc độ học** ở đầu **mỗi batch**. Nó áp dụng logic được mô tả trong sách: tăng tốc độ học tuyến tính trong khoảng một nửa thời gian huấn luyện, sau đó giảm nó tuyến tính trở lại tốc độ học ban đầu, và cuối cùng giảm nó xuống gần bằng không một cách tuyến tính trong phần rất cuối của quá trình huấn luyện.

In [ ]:
import math
import tensorflow as tf

class OneCycleScheduler(tf.keras.callbacks.Callback):
    def __init__(self, total_steps, max_lr=0.1, start_lr=None, last_lr=None):
        super().__init__()
        self.total_steps = total_steps
        self.max_lr = max_lr
        self.start_lr = start_lr or max_lr / 10
        self.last_lr = last_lr or self.start_lr / 1e4
        self.iteration = 0

    def lr_schedule(self, t):
        if t <= 0.3:
            return self.start_lr + t / 0.3 * (self.max_lr - self.start_lr)
        elif t <= 0.7:
            return self.max_lr - (t - 0.3) / 0.4 * (self.max_lr - self.start_lr)
        else:
            return self.start_lr - (t - 0.7) / 0.3 * (self.start_lr - self.last_lr)

    def on_batch_begin(self, batch, logs=None):
        t = self.iteration / self.total_steps
        lr = self.lr_schedule(t)
        # ✅ Sửa ở đây
        self.model.optimizer.learning_rate.assign(lr)
        self.iteration += 1

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['learning_rate'] = float(self.model.optimizer.learning_rate.numpy())


Hãy xây dựng và biên dịch một mô hình Fashion MNIST đơn giản, sau đó huấn luyện nó bằng cách sử dụng callback `OneCycleScheduler`:

In [ ]:
import math

model = build_model()
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.SGD(),
    metrics=["accuracy"]
)

n_epochs = 25
total_steps = math.ceil(len(X_train) / batch_size) * n_epochs
onecycle = OneCycleScheduler(total_steps, max_lr=0.1)

history = model.fit(
    X_train, y_train,
    epochs=n_epochs,
    batch_size=batch_size,
    validation_data=(X_valid, y_valid),
    callbacks=[onecycle]
)


# Tránh Hiện tượng Quá khớp Thông qua Điều chuẩn hóa (Regularization)

## Điều chuẩn hóa $\ell_1$ và $\ell_2$

In [ ]:
layer = tf.keras.layers.Dense(100, activation="relu",
                              kernel_initializer="he_normal",
                              kernel_regularizer=tf.keras.regularizers.l2(0.01))

Hoặc sử dụng `l1(0.1)` cho **điều chuẩn hóa $\ell_1$** với hệ số 0.1, hoặc `l1_l2(0.1, 0.01)` cho cả điều chuẩn hóa $\ell_1$ và $\ell_2$, với các hệ số lần lượt là 0.1 và 0.01.

In [ ]:
tf.random.set_seed(42)  # extra code – for reproducibility

In [ ]:
from functools import partial

RegularizedDense = partial(tf.keras.layers.Dense,
                           activation="relu",
                           kernel_initializer="he_normal",
                           kernel_regularizer=tf.keras.regularizers.l2(0.01))

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    RegularizedDense(100),
    RegularizedDense(100),
    RegularizedDense(10, activation="softmax")
])

In [ ]:
# extra code – compile and train the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.02)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=2,
                    validation_data=(X_valid, y_valid))

## Dropout

In [ ]:
tf.random.set_seed(42)  # extra code – for reproducibility

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(rate=0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [ ]:
# extra code – compile and train the model
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

Độ chính xác khi **huấn luyện (training accuracy)** có vẻ như thấp hơn độ chính xác khi **xác thực (validation accuracy)**, nhưng điều này chỉ là vì **dropout** chỉ hoạt động trong quá trình huấn luyện. Nếu chúng ta **đánh giá (evaluate)** mô hình trên tập huấn luyện sau khi huấn luyện (tức là với dropout đã **tắt**), chúng ta sẽ có được độ chính xác huấn luyện "thực sự", hơi cao hơn một chút so với độ chính xác xác thực và độ chính xác kiểm tra:

In [ ]:
model.evaluate(X_train, y_train)

In [ ]:
model.evaluate(X_test, y_test)

**Lưu ý**: hãy đảm bảo sử dụng **`AlphaDropout`** thay vì **`Dropout`** nếu bạn muốn xây dựng một **mạng nơ-ron tự chuẩn hóa (self-normalizing neural net)** sử dụng **SELU**.

## MC Dropout

In [ ]:
tf.random.set_seed(42)  # extra code – for reproducibility

In [ ]:
y_probas = np.stack([model(X_test, training=True)
                     for sample in range(100)])
y_proba = y_probas.mean(axis=0)

In [ ]:
model.predict(X_test[:1]).round(3)

In [ ]:
y_proba[0].round(3)

In [ ]:
y_std = y_probas.std(axis=0)
y_std[0].round(3)

In [ ]:
y_pred = y_proba.argmax(axis=1)
accuracy = (y_pred == y_test).sum() / len(y_test)
accuracy

In [ ]:
class MCDropout(tf.keras.layers.Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

In [ ]:
# extra code – shows how to convert Dropout to MCDropout in a Sequential model
Dropout = tf.keras.layers.Dropout
mc_model = tf.keras.Sequential([
    MCDropout(layer.rate) if isinstance(layer, Dropout) else layer
    for layer in model.layers
])
mc_model.set_weights(model.get_weights())

In [ ]:
mc_model.summary()

Bây giờ chúng ta có thể sử dụng mô hình với **MC Dropout**:

In [ ]:
# extra code – shows that the model works without retraining
tf.random.set_seed(42)
np.mean([mc_model.predict(X_test[:1])
         for sample in range(100)], axis=0).round(2)

## Chuẩn tối đa (Max norm)

In [ ]:
dense = tf.keras.layers.Dense(
    100, activation="relu", kernel_initializer="he_normal",
    kernel_constraint=tf.keras.constraints.max_norm(1.))

In [ ]:
# extra code – shows how to apply max norm to every hidden layer in a model

MaxNormDense = partial(tf.keras.layers.Dense,
                       activation="relu", kernel_initializer="he_normal",
                       kernel_constraint=tf.keras.constraints.max_norm(1.))

tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    MaxNormDense(100),
    MaxNormDense(100),
    tf.keras.layers.Dense(10, activation="softmax")
])
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

# Luyện tập

## 1. tới 7.

1. **Khởi tạo Glorot** và **Khởi tạo He** được thiết kế để làm cho độ lệch chuẩn đầu ra càng gần với độ lệch chuẩn đầu vào càng tốt, ít nhất là ở giai đoạn bắt đầu huấn luyện. Điều này làm giảm **vấn đề triệt tiêu/bùng nổ độ dốc**.
2. Không, tất cả các trọng số nên được **lấy mẫu một cách độc lập**; chúng không nên có cùng một giá trị khởi tạo. Một mục tiêu quan trọng của việc lấy mẫu trọng số ngẫu nhiên là để **phá vỡ tính đối xứng (break symmetry)**: nếu tất cả các trọng số có cùng một giá trị ban đầu, ngay cả khi giá trị đó không phải là zero, thì tính đối xứng sẽ không bị phá vỡ (tức là tất cả các nơ-ron trong một lớp nhất định đều tương đương), và **lan truyền ngược (backpropagation)** sẽ không thể phá vỡ nó. Cụ thể, điều này có nghĩa là tất cả các nơ-ron trong bất kỳ lớp nào cũng sẽ luôn có cùng một trọng số. Nó giống như chỉ có một nơ-ron trên mỗi lớp, và chậm hơn nhiều. Hầu như không thể để một cấu hình như vậy hội tụ về một giải pháp tốt.
3. Việc khởi tạo các **hệ số thiên vị (bias terms)** bằng zero là hoàn toàn ổn. Một số người thích khởi tạo chúng giống như trọng số, và điều đó cũng được; nó không tạo ra nhiều khác biệt.
4. **ReLU** thường là một lựa chọn mặc định tốt cho các lớp ẩn, vì nó nhanh và mang lại kết quả tốt. Khả năng xuất ra chính xác giá trị zero của nó cũng có thể hữu ích trong một số trường hợp (ví dụ, xem Chương 17). Hơn nữa, đôi khi nó có thể hưởng lợi từ các triển khai được tối ưu hóa cũng như từ sự tăng tốc phần cứng. Các biến thể **leaky ReLU** của ReLU có thể cải thiện chất lượng của mô hình mà không làm giảm tốc độ quá nhiều so với ReLU. Đối với các mạng nơ-ron lớn và các vấn đề phức tạp hơn, **GLU, Swish** và **Mish** có thể mang lại cho bạn một mô hình chất lượng cao hơn một chút, nhưng chúng có **chi phí tính toán (computational cost)**. **Hàm tang hyperbolic (tanh)** có thể hữu ích ở lớp đầu ra nếu bạn cần xuất ra một số trong một phạm vi cố định (mặc định là từ –1 đến 1), nhưng ngày nay nó không được sử dụng nhiều trong các lớp ẩn, ngoại trừ trong các mạng lặp (recurrent nets). **Hàm kích hoạt sigmoid** cũng hữu ích ở lớp đầu ra khi bạn cần ước tính một **xác suất** (ví dụ, cho phân loại nhị phân), nhưng nó hiếm khi được sử dụng trong các lớp ẩn (có những ngoại lệ—ví dụ, cho lớp mã hóa của các bộ **tự mã hóa biến phân (variational autoencoders)**; xem Chương 17). **Hàm kích hoạt softplus** hữu ích ở lớp đầu ra khi bạn cần đảm bảo rằng đầu ra sẽ luôn dương. **Hàm kích hoạt softmax** hữu ích ở lớp đầu ra để ước tính xác suất cho các lớp loại trừ lẫn nhau, nhưng nó hiếm khi (nếu có) được sử dụng trong các lớp ẩn.
5. Nếu bạn đặt **siêu tham số `momentum`** quá gần 1 (ví dụ: 0.99999) khi sử dụng trình tối ưu hóa **`SGD`**, thì thuật toán có thể sẽ đạt được tốc độ rất lớn, hy vọng là di chuyển gần về phía **cực tiểu toàn cục (global minimum)**, nhưng **động lượng** của nó sẽ mang nó vượt qua ngay cực tiểu đó. Sau đó, nó sẽ chậm lại và quay lại, tăng tốc trở lại, lại vượt quá mục tiêu, và cứ thế tiếp diễn. Nó có thể dao động theo cách này nhiều lần trước khi hội tụ, vì vậy nhìn chung sẽ mất nhiều thời gian hơn để hội tụ so với việc sử dụng giá trị `momentum` nhỏ hơn.
6. Một cách để tạo ra một **mô hình thưa thớt (sparse model)** (tức là với hầu hết các trọng số bằng zero) là huấn luyện mô hình bình thường, sau đó đặt các trọng số rất nhỏ bằng zero. Để có độ thưa thớt cao hơn, bạn có thể áp dụng **điều chuẩn hóa $\ell_1$** trong quá trình huấn luyện, điều này thúc đẩy trình tối ưu hóa hướng tới tính thưa thớt. Một lựa chọn thứ ba là sử dụng **TensorFlow Model Optimization Toolkit**.
7. Có, **dropout** làm chậm quá trình huấn luyện, nói chung là khoảng gấp đôi. Tuy nhiên, nó **không ảnh hưởng** đến tốc độ **suy luận (inference speed)** vì nó chỉ được bật trong quá trình huấn luyện. **MC Dropout** hoàn toàn giống như dropout trong quá trình huấn luyện, nhưng nó **vẫn hoạt động** trong quá trình suy luận, vì vậy mỗi lần suy luận bị chậm lại một chút. Quan trọng hơn, khi sử dụng MC Dropout, bạn thường muốn chạy suy luận **10 lần trở lên** để có được các dự đoán tốt hơn. Điều này có nghĩa là việc đưa ra dự đoán bị chậm lại gấp 10 lần hoặc hơn.

## 8. Học Sâu trên CIFAR10

### a.
*Bài tập: Xây dựng một **Mạng Nơ-ron Sâu (DNN)** với 20 lớp ẩn, mỗi lớp có 100 nơ-ron (quá nhiều, nhưng đó là mục đích của bài tập này). Sử dụng **khởi tạo He** và **hàm kích hoạt Swish**.*

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100,
                                    activation="swish",
                                    kernel_initializer="he_normal"))

### b.
*Bài tập: Sử dụng **tối ưu hóa Nadam** và **dừng sớm (early stopping)**, huấn luyện mạng trên tập dữ liệu **CIFAR10**. Bạn có thể tải nó bằng `tf.keras.datasets.cifar10.load_data()`. Tập dữ liệu bao gồm 60.000 hình ảnh màu 32 × 32 pixel (50.000 cho huấn luyện, 10.000 cho kiểm tra) với 10 lớp, vì vậy bạn sẽ cần một **lớp đầu ra softmax** với 10 nơ-ron. Hãy nhớ tìm kiếm **tốc độ học phù hợp** mỗi khi bạn thay đổi kiến trúc hoặc siêu tham số của mô hình.*

Hãy thêm **lớp đầu ra** vào mô hình:

In [ ]:
model.add(tf.keras.layers.Dense(10, activation="softmax"))

Hãy sử dụng **bộ tối ưu hóa Nadam** với **tốc độ học** là $5e-5$. Tôi đã thử các tốc độ học $1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3$ và $1e-2$, và tôi đã so sánh các **đường cong học tập (learning curves)** của chúng cho mỗi 10 **epoch** (sử dụng callback **TensorBoard**, ở dưới). Các tốc độ học $3e-5$ và $1e-4$ khá tốt, vì vậy tôi đã thử $5e-5$, kết quả là hơi tốt hơn một chút.

In [ ]:
optimizer = tf.keras.optimizers.Nadam(learning_rate=5e-5)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

Hãy tải tập dữ liệu **CIFAR10**. Chúng ta cũng muốn sử dụng **dừng sớm (early stopping)**, vì vậy chúng ta cần một **tập xác thực (validation set)**. Hãy sử dụng 5.000 hình ảnh đầu tiên của tập huấn luyện gốc làm tập xác thực:

In [ ]:
cifar10 = tf.keras.datasets.cifar10.load_data()
(X_train_full, y_train_full), (X_test, y_test) = cifar10

X_train = X_train_full[5000:]
y_train = y_train_full[5000:]
X_valid = X_train_full[:5000]
y_valid = y_train_full[:5000]

Bây giờ chúng ta có thể tạo các **callback** cần thiết và huấn luyện mô hình:

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=20,
                                                     restore_best_weights=True)
model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("my_cifar10_model",
                                                         save_best_only=True)
run_index = 1 # increment every time you train the model
run_logdir = Path() / "my_cifar10_logs" / f"run_{run_index:03d}"
tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)
callbacks = [early_stopping_cb, model_checkpoint_cb, tensorboard_cb]

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./my_cifar10_logs

In [ ]:
model.fit(X_train, y_train, epochs=100,
          validation_data=(X_valid, y_valid),
          callbacks=callbacks)

In [ ]:
model.evaluate(X_valid, y_valid)

Mô hình có **độ lỗi xác thực (validation loss)** thấp nhất đạt độ chính xác khoảng **46.8%** trên tập xác thực. Phải mất 29 **epoch** để đạt được độ lỗi xác thực thấp nhất, với khoảng 10 giây mỗi epoch trên máy tính xách tay của tôi (không có **GPU**). Hãy xem liệu chúng ta có thể cải thiện mô hình bằng cách sử dụng **Chuẩn hóa Hàng loạt (Batch Normalization)**.

### c.
*Bài tập: Bây giờ hãy thử thêm **Chuẩn hóa Hàng loạt (Batch Normalization)** và so sánh các **đường cong học tập (learning curves)**: Nó có **hội tụ nhanh hơn** trước không? Nó có tạo ra một **mô hình tốt hơn** không? Nó ảnh hưởng đến **tốc độ huấn luyện** như thế nào?*

Đoạn mã dưới đây rất giống với đoạn mã ở trên, với một vài thay đổi:

* Tôi đã thêm một lớp **BN (Batch Normalization)** sau mỗi lớp **Dense** (trước hàm kích hoạt), ngoại trừ lớp đầu ra.
* Tôi đã thay đổi **tốc độ học** thành $5e-4$. Tôi đã thử nghiệm với các giá trị $1e-5, 3e-5, 5e-5, 1e-4, 3e-4, 5e-4, 1e-3$ và $3e-3$, và tôi đã chọn giá trị có **hiệu suất xác thực** tốt nhất sau 20 **epoch**.
* Tôi đã đổi tên các thư mục chạy thành `run_bn_*` và tên tệp mô hình thành `my_cifar10_bn_model`.

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100, kernel_initializer="he_normal"))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.Activation("swish"))

model.add(tf.keras.layers.Dense(10, activation="softmax"))

optimizer = tf.keras.optimizers.Nadam(learning_rate=5e-4)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=20,
                                                     restore_best_weights=True)
model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("my_cifar10_bn_model",
                                                         save_best_only=True)
run_index = 1 # increment every time you train the model
run_logdir = Path() / "my_cifar10_logs" / f"run_bn_{run_index:03d}"
tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)
callbacks = [early_stopping_cb, model_checkpoint_cb, tensorboard_cb]

model.fit(X_train, y_train, epochs=100,
          validation_data=(X_valid, y_valid),
          callbacks=callbacks)

model.evaluate(X_valid, y_valid)

* *Mô hình có **hội tụ nhanh hơn** trước không?* **Nhanh hơn nhiều!** Mô hình trước đó mất 29 **epoch** để đạt được độ lỗi xác thực thấp nhất, trong khi mô hình mới đạt được cùng mức độ lỗi đó chỉ trong **12 epoch** và tiếp tục có tiến triển cho đến epoch thứ 17. Các lớp **BN (Batch Normalization)** đã **ổn định quá trình huấn luyện** và cho phép chúng ta sử dụng **tốc độ học lớn hơn** nhiều, do đó sự hội tụ nhanh hơn.
* *BN có tạo ra **mô hình tốt hơn** không?* **Có!** Mô hình cuối cùng cũng tốt hơn nhiều, với độ chính xác xác thực là **50.7%** thay vì 46.7%. Nó vẫn chưa phải là một mô hình rất tốt, nhưng ít nhất thì nó tốt hơn nhiều so với trước đây (**Mạng Nơ-ron Tích chập - Convolutional Neural Network** sẽ làm tốt hơn nhiều, nhưng đó là một chủ đề khác, xem chương 14).
* *BN ảnh hưởng đến **tốc độ huấn luyện** như thế nào?* Mặc dù mô hình hội tụ nhanh hơn nhiều, mỗi **epoch** mất khoảng **15 giây** thay vì 10 giây, do các phép tính bổ sung cần thiết của các lớp BN. Nhưng nhìn chung **thời gian huấn luyện (wall time)** để đạt được mô hình tốt nhất đã được **rút ngắn khoảng 10%**.

### d.
*Bài tập: Hãy thử thay thế **Chuẩn hóa Hàng loạt (Batch Normalization)** bằng **SELU**, và thực hiện các điều chỉnh cần thiết để đảm bảo mạng **tự chuẩn hóa (self-normalizes)** (tức là **chuẩn hóa** các **đặc trưng đầu vào (input features)**, sử dụng **khởi tạo bình thường LeCun**, đảm bảo **Mạng Nơ-ron Sâu (DNN)** chỉ chứa một chuỗi các lớp **dense**, v.v.).*

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100,
                                    kernel_initializer="lecun_normal",
                                    activation="selu"))

model.add(tf.keras.layers.Dense(10, activation="softmax"))

optimizer = tf.keras.optimizers.Nadam(learning_rate=7e-4)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=20, restore_best_weights=True)
model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "my_cifar10_selu_model", save_best_only=True)
run_index = 1 # increment every time you train the model
run_logdir = Path() / "my_cifar10_logs" / f"run_selu_{run_index:03d}"
tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)
callbacks = [early_stopping_cb, model_checkpoint_cb, tensorboard_cb]

X_means = X_train.mean(axis=0)
X_stds = X_train.std(axis=0)
X_train_scaled = (X_train - X_means) / X_stds
X_valid_scaled = (X_valid - X_means) / X_stds
X_test_scaled = (X_test - X_means) / X_stds

model.fit(X_train_scaled, y_train, epochs=100,
          validation_data=(X_valid_scaled, y_valid),
          callbacks=callbacks)

model.evaluate(X_valid_scaled, y_valid)

Mô hình này đạt đến **độ lỗi xác thực** của mô hình đầu tiên chỉ trong **8 epoch**. Sau 14 epoch, nó đạt đến **độ lỗi xác thực thấp nhất**, với độ chính xác khoảng **50.3%**, tốt hơn so với mô hình gốc (46.7%), nhưng không bằng mô hình sử dụng **chuẩn hóa hàng loạt (batch normalization)** (50.7%). Mỗi epoch chỉ mất **9 giây**. Vì vậy, đây là mô hình **huấn luyện nhanh nhất** cho đến nay.

### e.
*Bài tập: Hãy thử **điều chuẩn hóa** mô hình bằng **alpha dropout**. Sau đó, **mà không cần huấn luyện lại** mô hình, hãy xem liệu bạn có thể đạt được độ chính xác tốt hơn bằng cách sử dụng **MC Dropout**.*

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100,
                                    kernel_initializer="lecun_normal",
                                    activation="selu"))

model.add(tf.keras.layers.AlphaDropout(rate=0.1))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

optimizer = tf.keras.optimizers.Nadam(learning_rate=5e-4)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    patience=20, restore_best_weights=True)
model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "my_cifar10_alpha_dropout_model", save_best_only=True)
run_index = 1 # increment every time you train the model
run_logdir = Path() / "my_cifar10_logs" / f"run_alpha_dropout_{run_index:03d}"
tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)
callbacks = [early_stopping_cb, model_checkpoint_cb, tensorboard_cb]

X_means = X_train.mean(axis=0)
X_stds = X_train.std(axis=0)
X_train_scaled = (X_train - X_means) / X_stds
X_valid_scaled = (X_valid - X_means) / X_stds
X_test_scaled = (X_test - X_means) / X_stds

model.fit(X_train_scaled, y_train, epochs=100,
          validation_data=(X_valid_scaled, y_valid),
          callbacks=callbacks)

model.evaluate(X_valid_scaled, y_valid)

Mô hình đạt độ chính xác **48.1%** trên **tập xác thực (validation set)**. Kết quả này **tệ hơn** so với khi không có **dropout** (50.3%). Với một quá trình tìm kiếm **siêu tham số (hyperparameter search)** sâu rộng, có thể đạt được kết quả tốt hơn (tôi đã thử các tỷ lệ dropout 5%, 10%, 20% và 40%, cùng với các tốc độ học $1e-4, 3e-4, 5e-4$ và $1e-3$), nhưng có lẽ trong trường hợp này cũng không thể tốt hơn nhiều.

Bây giờ chúng ta hãy sử dụng **MC Dropout**. Chúng ta sẽ cần lớp **`MCAlphaDropout`** mà chúng ta đã sử dụng trước đó, vì vậy hãy sao chép nó vào đây để tiện lợi:

In [ ]:
class MCAlphaDropout(tf.keras.layers.AlphaDropout):
    def call(self, inputs):
        return super().call(inputs, training=True)

Bây giờ chúng ta hãy tạo một mô hình mới, **giống hệt** mô hình chúng ta vừa huấn luyện (với cùng **trọng số**), nhưng với các lớp **dropout** là **`MCAlphaDropout`** thay vì các lớp **`AlphaDropout`**:

In [ ]:
mc_model = tf.keras.Sequential([
    (
        MCAlphaDropout(layer.rate)
        if isinstance(layer, tf.keras.layers.AlphaDropout)
        else layer
    )
    for layer in model.layers
])

Sau đó, chúng ta hãy thêm một vài **hàm tiện ích (utility functions)**. Hàm đầu tiên sẽ chạy mô hình nhiều lần (mặc định là 10) và nó sẽ trả về **xác suất lớp được dự đoán trung bình (mean predicted class probabilities)**. Hàm thứ hai sẽ sử dụng các xác suất trung bình này để dự đoán **lớp có khả năng nhất** cho mỗi cá thể:

In [ ]:
def mc_dropout_predict_probas(mc_model, X, n_samples=10):
    Y_probas = [mc_model.predict(X) for sample in range(n_samples)]
    return np.mean(Y_probas, axis=0)

def mc_dropout_predict_classes(mc_model, X, n_samples=10):
    Y_probas = mc_dropout_predict_probas(mc_model, X, n_samples)
    return Y_probas.argmax(axis=1)

Bây giờ chúng ta hãy đưa ra **dự đoán** cho tất cả các cá thể trong **tập xác thực (validation set)**, và tính toán **độ chính xác**:

In [ ]:
tf.random.set_seed(42)

y_pred = mc_dropout_predict_classes(mc_model, X_valid_scaled)
accuracy = (y_pred == y_valid[:, 0]).mean()
accuracy

Chúng ta quay trở lại với độ chính xác xấp xỉ của mô hình không có **dropout** trong trường hợp này (độ chính xác khoảng 50.3%).

Vì vậy, mô hình tốt nhất chúng ta có được trong bài tập này là mô hình **Chuẩn hóa Hàng loạt (Batch Normalization)**.

### f.
*Bài tập: **Huấn luyện lại** mô hình của bạn bằng cách sử dụng **lập lịch 1 chu kỳ (1cycle scheduling)** và xem liệu nó có **cải thiện tốc độ huấn luyện** và **độ chính xác của mô hình** hay không.*

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100,
                                    kernel_initializer="lecun_normal",
                                    activation="selu"))

model.add(tf.keras.layers.AlphaDropout(rate=0.1))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

optimizer = tf.keras.optimizers.SGD()
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

In [ ]:
batch_size = 128
rates, losses = find_learning_rate(model, X_train_scaled, y_train, epochs=1,
                                   batch_size=batch_size)
plot_lr_vs_loss(rates, losses)

In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Flatten(input_shape=[32, 32, 3]))
for _ in range(20):
    model.add(tf.keras.layers.Dense(100,
                                 kernel_initializer="lecun_normal",
                                 activation="selu"))

model.add(tf.keras.layers.AlphaDropout(rate=0.1))
model.add(tf.keras.layers.Dense(10, activation="softmax"))

optimizer = tf.keras.optimizers.SGD(learning_rate=2e-2)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

In [ ]:
n_epochs = 15
n_iterations = math.ceil(len(X_train_scaled) / batch_size) * n_epochs
onecycle = OneCycleScheduler(n_iterations, max_lr=0.05)
history = model.fit(X_train_scaled, y_train, epochs=n_epochs, batch_size=batch_size,
                    validation_data=(X_valid_scaled, y_valid),
                    callbacks=[onecycle])

**Một chu kỳ (One cycle)** đã cho phép chúng ta huấn luyện mô hình chỉ trong **15 epoch**, mỗi epoch chỉ mất **2 giây** (nhờ **kích thước batch lớn hơn**). Điều này **nhanh hơn nhiều lần** so với mô hình nhanh nhất mà chúng ta đã huấn luyện cho đến nay. Hơn nữa, chúng ta đã **cải thiện hiệu suất của mô hình** (từ 50.7% lên 52.0%).